# Review TF-IDF candidates for our learning-goal queries

**Aim:** See the first ten books returned for each of the 30 reviewed queries and inspect the evidence needed to label them. No relevance labels are assigned by this notebook.

The unchanged baseline 0 uses TF-IDF defaults and cosine similarity. Only the `query` string drives retrieval. `background` and `success_criterion` are shown as judgment context; they are not appended to the query.

| Section | What we do |
| --- | --- |
| [1. Load](#section-1) | Identify the frozen candidate run and its source files. |
| [2. Select a query](#section-2) | Show its background, criterion, and ten candidate books. |
| [3. Inspect one book](#section-3) | Read its saved description, features, and prepared TOC. |
| [4. Validate](#section-4) | Verify all 300 candidates against the saved complete rankings. |

The [all-query report](../benchmarks/gold_v1/tfidf_candidates.md) lists every candidate without running Python. Create a new local run with `uv run python -m bookpath.candidates` if none exists. Existing output folders are protected.

**A grade belongs to a query–book pair, not a retrieval method.** Several books can receive 3. New methods' unjudged results need review before fair comparison. Add their judgments to a versioned shared pool and recalculate all methods against the same version.

<a id="section-1"></a>

## 1. What files supply the candidates and book evidence?

**Objective:** Load the frozen run and the original metadata. The raw file provides description and features; the prepared TOC file supplies the text and chapter metadata reviewed in notebook 03.

In [1]:
import hashlib
import html
import json
from pathlib import Path

import pandas as pd
from IPython.display import HTML, Markdown, display

project_root = Path.cwd()
while not (project_root / "src/bookpath").is_dir():
    if project_root == project_root.parent:
        raise FileNotFoundError("Open this notebook inside the BookPath repository.")
    project_root = project_root.parent

run_path = project_root / "data/evaluation/gold_v1_tfidf/candidates.json"
rankings_path = run_path.with_name("rankings.parquet")
queries_path = project_root / "benchmarks/gold_v1/queries.json"
documents_path = project_root / "data/processed/documents.parquet"
raw_path = project_root / "data/raw/books.parquet"
toc_path = project_root / "data/processed/toc_features.parquet"
if not run_path.exists():
    raise FileNotFoundError("Run uv run python -m bookpath.candidates first.")


def checksum(path):
    with path.open("rb") as source:
        return hashlib.file_digest(source, "sha256").hexdigest()


paths = [run_path, rankings_path, queries_path, documents_path, raw_path, toc_path]
checksums_before = {path: checksum(path) for path in paths}
run = json.loads(run_path.read_text())
query_set = json.loads(queries_path.read_text())
rankings = pd.read_parquet(rankings_path)
books = pd.read_parquet(raw_path)
toc_features = pd.read_parquet(toc_path)
print(f"{run['query_count']} queries; {run['candidate_books_per_query']:,} books ranked per query.")
print(f"{run['full_ranking_rows']:,} full ranking rows; {run['review_pairs']} top-ten review pairs.")
print(run["labels_status"])
display(Markdown("**Conclusion:** We have TF-IDF candidates for all queries, with no relevance grades. This is the first source for the eventual candidate pool."))

30 queries; 9,034 books ranked per query.
271,020 full ranking rows; 300 top-ten review pairs.
No relevance labels assigned; TF-IDF candidates only, not a complete pool.


**Conclusion:** We have TF-IDF candidates for all queries, with no relevance grades. This is the first source for the eventual candidate pool.

<a id="section-2"></a>

## 2. Which books does TF-IDF return for this query?

**Objective:** Choose a query by its number (1–30), then inspect its top ten. Change `query_number` and rerun this section and section 3.

`show_rank_and_scores = True` shows the retrieval order. Set it to `False` when grading: books are then displayed alphabetically, with ranks and scores hidden. This changes only the display, not the saved ranking.

In [2]:
query_number = 1  # Choose a number from 1 to 30.
show_rank_and_scores = True

if not 1 <= query_number <= len(run["queries"]):
    raise ValueError("Choose an existing query number.")
selected_query = run["queries"][query_number - 1]
print("Query ID:", selected_query["query_id"])
print("Background:", selected_query["background"])
print("Query:", selected_query["query"])
print("Success criterion:", selected_query["success_criterion"])
candidates = pd.DataFrame(selected_query["books"])
if show_rank_and_scores:
    shown = candidates
else:
    shown = candidates[["parent_asin", "book_title"]].sort_values("book_title")
with pd.option_context("display.max_colwidth", None, "display.float_format", "{:.6f}".format):
    display(shown)
display(Markdown("**Conclusion:** These ten books were selected by cosine similarity. Their relevance remains for you to judge from the learning goal and book evidence."))

Query ID: online_business_finance_001
Background: Finance professional
Query: I want to know what is needed to start a online business, given that i  have a finance background
Success criterion: Practical guidance on starting an online business; finance experience is context, not a request for investment advice.


,rank,parent_asin,book_title,cosine_similarity
0,1,1683509854,Don’t Sleep on It: Turn Your Passion & Expertise into a Profitable Online Business,0.261789
1,2,1705797547,The 60 Minute Startup: A Proven System to Start Your Business in 1 Hour a Day and Get Your First Paying Customers in 30 Days (or Less) (The Agile Entrepreneurship Series),0.240739
2,3,1119228352,Investing Online For Dummies,0.234915
3,4,0470769262,Investing Online For Dummies,0.232961
4,5,1118495365,Investing Online For Dummies,0.230588
5,6,0793145171,Selling Online: How to Become a Successful E-Commerce Merchant,0.229723
6,7,1581159021,The Pocket Small Business Owner's Guide to Building Your Business (Pocket Small Business Owner's Guides),0.229596
7,8,1118926706,Starting an Online Business All-in-One For Dummies,0.219628
8,9,0764503359,Marketing Online For Dummies,0.205789
9,10,0538496894,Entrepreneurship: Ideas in Action (DECA),0.203651


**Conclusion:** These ten books were selected by cosine similarity. Their relevance remains for you to judge from the learning goal and book evidence.

<a id="section-3"></a>

## 3. What evidence do we have for one candidate?

**Objective:** Inspect a book before judging it. Enter a quoted `parent_asin` from the candidate table. `None` displays the first book in the current view.

`description` and `features` are original book-level columns. The TOC table shows prepared `base_text` and `chapter_subtitle` from the raw TOC items, with `raw_position` for traceability. We show both description and features for judgment, even though retrieval used features with description only as a fallback. An empty field is shown explicitly. These are catalog records, not the complete book.

In [3]:
example_asin = None  # Or a quoted Book ID from the candidate table.
if example_asin is None:
    example_asin = shown.iloc[0]["parent_asin"]
if example_asin not in set(candidates["parent_asin"]):
    raise ValueError("Choose a Book ID from this query's candidate list.")
selected_book = books.loc[books["parent_asin"].eq(example_asin)].iloc[0]
print(selected_book["title"], "parent_asin =", example_asin)

for field in ["description", "features"]:
    pieces = list(selected_book[field])
    print(f"Original {field}: {len(pieces)} saved pieces.")
    if not pieces:
        print("No saved content in this field.")
        continue
    text = "\n".join(pieces)
    print(text[:800])
    display(HTML(
        f"<details><summary>Read complete {field}</summary>"
        f"<div style='max-width: 90ch; white-space: pre-wrap'>{html.escape(text)}</div></details>"
    ))

book_toc = toc_features.loc[toc_features["parent_asin"].eq(example_asin)]
print(f"Prepared TOC: {len(book_toc)} original-item rows.")
with pd.option_context("display.max_colwidth", None):
    display(book_toc[["raw_position", "base_text", "chapter_subtitle"]].reset_index(drop=True))
display(Markdown("**Conclusion:** Review this evidence against the selected query. Use 0–3 for relevance or leave the book unjudged when evidence is insufficient. Changing the selected book does not edit results or labels."))

Don’t Sleep on It: Turn Your Passion & Expertise into a Profitable Online Business parent_asin = 1683509854
Original description: 2 saved pieces.
About the Author
Having immersed himself in learning about artist promotion after learning to play the Tabla, Kavit Haria performed across the UK with many artists (including Jimmy Page at the Royal Albert Hall) and recorded albums at the famed Abbey Road studios. Recognizing the demand for sharing what he learned gave rise to Kavit’s first company, Insider Music Business. Within five years (2005 to 2010) Kavit created 35 different educational products online. Kavit then turned to share his marketing automation strategies at business conferences in the UK and ran private workshops for small business owners in New York, Sydney, and London. Since 2013, Kavit has been running Automated Business System, a flagship service to help experts and brands strategize, build, launch and market their on


Original features: 3 saved pieces.
Don’t Sleep On It!
is a call to action for people to wake up and live their best life by taking what they know and who they are and unleashing it in the world, all while creating their very own profitable online business. So many people want to start a business, but don’t know what to do. With 10 years of online building experience, Kavit Haria guides readers through the steps to turn their passions and interests―their expertise and knowledge―into a profitable online business.
When you light a fire in someone else, you can change the world. Whether you know it or not, what you know at this time is what someone else needs in order to overcome their challenges, remedy their pain, or solve their problem. Stop trading time for money, and start sharing what you know and earning from your own on


Prepared TOC: 3 original-item rows.


,raw_position,base_text,chapter_subtitle
0,1,Part one. Business fundamentals,<NA>
1,2,Part two. Leveraging your expertise,<NA>
2,3,Part three. Marketing your course.,<NA>


**Conclusion:** Review this evidence against the selected query. Use 0–3 for relevance or leave the book unjudged when evidence is insufficient. Changing the selected book does not edit results or labels.

<a id="section-4"></a>

## 4. Do the review lists match the recorded retrieval run?

**Objective:** Verify all candidates, not just the selected query. Check input and code identities, each query's complete book coverage, all top-ten entries, and unchanged files. No new retrieval or labeling happens here.

In [4]:
document_sources = pd.read_parquet(documents_path, columns=["raw_source_sha256", "toc_source_sha256"])
assert document_sources["raw_source_sha256"].eq(checksums_before[raw_path]).all()
assert document_sources["toc_source_sha256"].eq(checksums_before[toc_path]).all()
assert checksums_before[documents_path] == run["documents_sha256"]
assert checksums_before[queries_path] == run["queries_sha256"]
assert len(run["queries"]) == len(query_set["queries"]) == run["query_count"]
assert len(rankings) == run["full_ranking_rows"]
assert not rankings.duplicated(["query_id", "parent_asin"]).any()
assert set(rankings["parent_asin"]) == set(books["parent_asin"])
review_pairs = 0
for record, query_record in zip(run["queries"], query_set["queries"], strict=True):
    for name, value in query_record.items():
        assert record[name] == value
    query_ranking = rankings.loc[rankings["query_id"].eq(record["query_id"])]
    assert len(query_ranking) == run["candidate_books_per_query"]
    assert query_ranking["rank"].tolist() == list(range(1, len(query_ranking) + 1))
    assert query_ranking.head(run["top_k"]).drop(columns="query_id").to_dict(orient="records") == record["books"]
    review_pairs += len(record["books"])
assert review_pairs == run["review_pairs"]
for name, value in run["code_sha256"].items():
    assert checksum(project_root / "src/bookpath" / name) == value
for path, value in checksums_before.items():
    assert checksum(path) == value
print(f"Validated all {review_pairs} candidate entries against {len(rankings):,} saved ranking rows.")
display(Markdown("**Conclusion:** The displayed candidates match the recorded queries, code, and complete rankings. Files are unchanged; relevance labels and quality metrics are still pending."))

Validated all 300 candidate entries against 271,020 saved ranking rows.


**Conclusion:** The displayed candidates match the recorded queries, code, and complete rankings. Files are unchanged; relevance labels and quality metrics are still pending.

**How new methods affect labels:** Suppose books A and B both fully meet the same query's criteria. Both can receive grade 3, even if different methods retrieve them. Precision and nDCG can reward either book. Recall measures how many known relevant books were found; it does not require one particular method's favorite to appear first.

If a new method finds book C, review C and add its label to a new pool version. Score every method against that same version. If C is better in a way the existing 0–3 rubric cannot distinguish, record the limitation or revise the rubric for all methods; do not change A's grade just because another method found C.

No candidates from other retrieval sources have been added yet. These are 300 TF-IDF query–book pairs for review, not a completed gold-standard evaluation set.

**Next task:** Start with query numbers 1, 7, 11, 16, and 24: 50 query–book pairs. Hide scores with `show_rank_and_scores = False`, inspect the book evidence, then send a grade of 0–3 and a short reason for each pair. Use `uncertain` when evidence is insufficient. The notebook does not save labels; the [labeling workflow](../benchmarks/gold_v1/README.md#your-next-task) explains the handoff.

**Sources:** Stanford explains [relevance judgments and pooling](https://nlp.stanford.edu/IR-book/html/htmledition/assessing-relevance-1.html), [test collections](https://nlp.stanford.edu/IR-book/html/htmledition/standard-test-collections-1.html), and [evaluation of ranked results](https://nlp.stanford.edu/IR-book/html/htmledition/evaluation-of-ranked-retrieval-results-1.html). The 0–3 rubric is our project choice. See the complete [project source index](../docs/references.md).